# Sliding Window Evaluation — Free BPB Win

This is one of the easiest improvements in the challenge: **change how you evaluate, not how you train**.

The baseline evaluates with non-overlapping chunks. Sliding window evaluation gives each token much more context, improving BPB by ~0.034 for free.

## 1. The Problem: Non-Overlapping Evaluation

The baseline splits validation text into non-overlapping chunks of 1024 tokens:

```
Chunk 1: [token_0, token_1, ..., token_1023]  → predict token_1 to token_1024
Chunk 2: [token_1024, token_1025, ..., token_2047]  → predict token_1025 to token_2048
...
```

**The problem:** Token_1024 (first token of chunk 2) has **zero context** — the model has to predict it cold.
Token_1025 has only 1 token of context. Token_1026 has 2 tokens. And so on.

The first ~100 tokens of each chunk are essentially wasted — the model can't predict them well because it hasn't seen enough context. This drags up the average loss.

In [ ]:
import sys
sys.path.insert(0, '../parameter-golf')

import numpy as np

# Illustrate the context problem
seq_len = 1024
n_chunks = 5

print("=== Non-Overlapping Evaluation ===")
print(f"Sequence length: {seq_len}")
print(f"")
for chunk in range(n_chunks):
    start = chunk * seq_len
    end = (chunk + 1) * seq_len
    positions_with_full_context = seq_len - 1  # position 1023 has 1023 tokens of context
    positions_with_no_context = 1  # position 0
    avg_context = (seq_len - 1) / 2
    print(f"  Chunk {chunk}: tokens [{start:>6d}, {end:>6d})  "
          f"context: 0→{seq_len-1}  avg_context: {avg_context:.0f}")

print(f"\nToken at position 0 of each chunk: 0 context tokens")
print(f"Token at position 512: 512 context tokens")
print(f"Token at position 1023: 1023 context tokens")
print(f"\nAverage context per token: {(seq_len-1)/2:.0f} tokens")
print(f"This is MUCH less than the max {seq_len-1}!")

## 2. The Solution: Sliding Window

Instead of non-overlapping chunks, slide a window with a small **stride** (e.g., 64 tokens):

```
Window 1: [token_0, ..., token_1023]    → score tokens 960-1023 (last 64)
Window 2: [token_64, ..., token_1087]   → score tokens 1024-1087 (last 64)
Window 3: [token_128, ..., token_1151]  → score tokens 1088-1151 (last 64)
...
```

Each token is scored only when it has **at least (seq_len - stride) = 960 tokens** of context.
That's way better than the 0-1023 range in non-overlapping eval!

In [ ]:
stride = 64

print(f"=== Sliding Window Evaluation (stride={stride}) ===")
print(f"Window size: {seq_len}")
print(f"Stride: {stride}")
print(f"Tokens scored per window: {stride} (only the last {stride} tokens)")
print(f"Min context for scored tokens: {seq_len - stride} = {seq_len - stride}")
print(f"")

n_windows = 5
for w in range(n_windows):
    win_start = w * stride
    win_end = win_start + seq_len
    score_start = win_end - stride
    score_end = win_end
    min_ctx = seq_len - stride
    print(f"  Window {w}: [{win_start:>6d}, {win_end:>6d})  "
          f"score [{score_start:>6d}, {score_end:>6d})  min_context: {min_ctx}")

print(f"\n=== Comparison ===")
print(f"Non-overlapping: avg context = {(seq_len-1)/2:.0f} tokens")
print(f"Sliding (s={stride}): min context = {seq_len - stride} tokens")
print(f"\nEvery scored token has at least {seq_len - stride} tokens of context!")
print(f"This is ~{(seq_len-stride) / ((seq_len-1)/2):.1f}x more context on average.")

## 3. The Cost: More Compute

The trade-off is clear: sliding window runs the model many more times.

In [ ]:
# Compute overhead for different strides
total_tokens = 62_000_000  # approximate val set size

print(f"Total validation tokens: {total_tokens:,}")
print(f"Sequence length: {seq_len}")
print(f"")
print(f"{'Stride':>8s}  {'Windows':>10s}  {'Overhead':>10s}  {'Min Context':>12s}  {'Est. Time':>10s}")
print("-" * 60)

non_overlap_windows = total_tokens // seq_len
for s in [seq_len, 512, 256, 128, 64, 32]:
    n_windows = (total_tokens - seq_len) // s + 1
    overhead = n_windows / non_overlap_windows
    min_ctx = seq_len - s
    est_time_s = n_windows / non_overlap_windows * 30  # baseline ~30s
    stride_label = f"{s}" if s < seq_len else f"{s} (none)"
    print(f"{stride_label:>8s}  {n_windows:>10,}  {overhead:>9.1f}x  {min_ctx:>12d}  {est_time_s:>8.0f}s")

print(f"\nStride 64 is the sweet spot: ~16x more compute but still under 10 min eval budget.")
print(f"The BPB improvement (~0.034) is bigger than many architectural changes!")

## 4. Implementation Sketch

Here's how to implement sliding window eval. This will be Experiment 01.

```python
def eval_val_sliding(model, val_tokens, seq_len=1024, stride=64):
    """Evaluate with sliding window for better BPB."""
    total_loss_sum = 0.0
    total_scored_tokens = 0
    total_bytes = 0.0
    
    for start in range(0, len(val_tokens) - seq_len, stride):
        # Get window
        window = val_tokens[start : start + seq_len + 1]
        x = window[:-1]  # input
        y = window[1:]   # target
        
        # Forward pass
        logits = model(x)
        losses = cross_entropy(logits, y)  # per-token losses
        
        # Only score the LAST `stride` tokens (they have full context)
        scored_losses = losses[-stride:]
        scored_tokens = y[-stride:]
        
        total_loss_sum += scored_losses.sum()
        total_scored_tokens += stride
        total_bytes += compute_bytes(scored_tokens)
    
    val_loss = total_loss_sum / total_scored_tokens
    val_bpb = (val_loss / log(2)) * (total_scored_tokens / total_bytes)
    return val_loss, val_bpb
```

The key insight: **only score the last `stride` tokens per window**, where context is maximal.

## 5. Why It's "Free"

This improvement:
- Does NOT change the model (same weights, same training)
- Does NOT change the training procedure
- Does NOT use more of the 16MB artifact budget
- Only changes evaluation — which has its own 10-minute budget

It's measuring the *same model* more fairly. With non-overlapping eval, we're penalizing the model for something it can't help (zero context at chunk boundaries).

**Expected improvement:** ~0.034 BPB (this is consistent across all top submissions)

## Key Takeaways

1. Non-overlapping eval unfairly penalizes tokens with little context
2. Sliding window with stride=64 gives every token ~960 tokens of context
3. Cost: ~16x more eval compute (still fits in 10-min eval budget on 8xH100)
4. Gain: ~0.034 BPB — bigger than many model architecture changes
5. This should be Experiment 01 — it's the easiest win in the challenge